In [ ]:
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
from itertools import combinations
from rapidfuzz import fuzz

In [ ]:
seed = 99
filepath_syn = Path(f"../data/synthetic/{seed}")
filepath_gt = Path(f"../data/ground_truth/{seed}")

customers_raw = pd.read_csv(filepath_syn/"customers_raw.csv")
products_raw = pd.read_csv(filepath_syn/"products_raw.csv")
orders_raw = pd.read_csv(filepath_syn/"orders_raw.csv")

customers_clean = pd.read_csv(filepath_gt/"customers_clean.csv")
products_clean = pd.read_csv(filepath_gt/"products_clean.csv")

In [ ]:
def show_all(df, name=""):
    with pd.option_context("display.max_rows", None,
                           "display.max_columns", None,
                           "display.width", None):
        if name:
            print(f"=== {name}: {len(df)} rows ===")
        display(df)


def inspect_duplicates(df, subset=None, name=""):
    dups = df[df.duplicated(subset=subset, keep=False)]
    sort_cols = subset if subset else list(df.columns)
    with pd.option_context("display.max_rows", None,
                           "display.max_columns", None):
        print(f"=== {name}: {len(dups)} duplicate rows "
              f"(subset={subset or 'all columns'}) ===")
        display(dups.sort_values(by=sort_cols))

def inspect_fuzzy_duplicates(df, cols=None, exclude=None, threshold=85,
                             scorer=fuzz.ratio, name=""):
    exclude = set(exclude or [])
    if cols is None:                      
        cols = [c for c in df.columns if c not in exclude]
    sub = df[cols].fillna("").astype(str)
    recs = sub.to_dict("records")
    idx = df.index.tolist()

    rows = []
    for a, b in combinations(range(len(df)), 2):
        scores = {c: scorer(recs[a][c], recs[b][c]) for c in cols}
        avg = sum(scores.values()) / len(cols)
        if avg >= threshold:
            rows.append({"score": round(avg, 1), "idx_a": idx[a], "idx_b": idx[b],
                         **{f"{c}_a": recs[a][c] for c in cols},
                         **{f"{c}_b": recs[b][c] for c in cols}})

    result = (pd.DataFrame(rows).sort_values("score", ascending=False)
              .reset_index(drop=True))
    with pd.option_context("display.max_rows", None, "display.max_columns", None):
        print(f"=== {name}: {len(result)} fuzzy pairs "
              f"(threshold={threshold}, cols={cols}) ===")
        display(result)
    return result

In [ ]:
# Customers

show_all(customers_raw.sort_values("customer_id"), "customers_raw")             #sorted raw
show_all(customers_clean.sort_values("customer_id"), "customers_clean")         #sorted clean
#show_all(customers_raw, "customers_raw")                                        #unsorted
#show_all(customers_clean, "customers_clean")                                    #unsorted

# Duplicates

#inspect_duplicates(customers_raw, name="customers_raw")
inspect_duplicates(customers_clean, name="customers_clean")    # 0 expected

#inspect_fuzzy_duplicates(customers_raw, cols=["full_name", "email"], threshold=80, name="customers_raw");

In [ ]:
# Products

show_all(products_raw.sort_values("product_id"), "products_raw")     #sorted
show_all(products_clean.sort_values("product_id"), "products_clean") 
#show_all(products, "products")                               #unsorted

# Duplicates

inspect_duplicates(products_raw, name="products_raw")
#inspect_duplicates(products_clean, subset=["product_id"], name="products_clean")

In [ ]:
# Orders

show_all(orders.sort_values("order_id"), "orders")          #sorted
#show_all(orders, "orders")                                 #unsorted

# Duplicates

inspect_duplicates(orders, name="orders")
#inspect_duplicates(orders, subset=["order_id"], name="orders")